# Drop size vs time (d²-law)

Reads `results/measurements.csv` from `droplet_size_measurement_v2.ipynb` and turns it into
size-vs-time curves, one per tracked drop (`drop_id`).

**What it does**
1. Loads the CSV and keeps only trustworthy rows. It prints exactly what was removed and why.
2. Fits each drop to the **d²-law**, `D(t)² = D₀² − K·t`, and reports the evaporation constant
   **K (µm²/s)** per drop. → `results/size_vs_time/drop_fits.csv`
3. Figure 1: a filmstrip of one drop, plus D vs t and D² vs t for a few example drops (the
   same layout as the earlier glare-separation plot).
4. Figure 2: every drop at once, aligned at its fitted disappearance time, plus the spread of K
   per video.
5. Figure 3: a QC list of drops whose fit looks off. These are the first place to look for a
   wrong `drop_id` link.

**How to use:** edit the Config cell, then Run All. Nothing here re-measures images; it only
reads the CSV (and, for the filmstrip only, the original video).

In [ ]:
# ============================== CONFIG ==============================
import os
DATE_DIR      = r"F:\Imaging\2026.10.12"
CSV_PATH      = os.path.join(DATE_DIR, "results", "measurements.csv")
VIDEO_DIR     = os.path.join(DATE_DIR, "drops")        # only used for the filmstrip
OUT_DIR       = os.path.join(DATE_DIR, "results", "size_vs_time")

VIDEOS        = None            # None = every video in the CSV, or e.g. ["evap_30", "evap_31"]
SIZE_COL      = "D_eq_um"       # circle-fit diameter (primary). Alternatives: "mean_HV_um", "V_um"

# --- which rows / drops to trust ---------------------------------------
MIN_FRAMES    = 8      # a drop needs at least this many frames to be fitted / plotted
SHARPNESS_MIN = 1.5    # rows below this are out of focus (soft, oversized shadows: the ~40-47 um
                       # single-frame "drops" in evap_29 are all < 1.25). In-focus drops sit ~2-2.5.
DROP_FIT_FAILED = True # discard rows where the circle fit failed

# --- d2-law fit ----------------------------------------------------------
FIT_SKIP_FIRST = 0     # skip this many frames at the start of each drop (entry frame can be distorted)
FIT_MIN_D_UM   = 0.0   # ignore points smaller than this in the fit (the tiniest drops are ~4 px)
FIT_OUTLIER_K  = 3.5   # one round of outlier rejection: drop points whose D^2 residual > this x robust sigma.
                       # Catches single-frame dips (a bad edge fit on one frame), which are common in evap_30.
                       # Set to None to fit every point.

# --- figures --------------------------------------------------------------
EXAMPLE_VIDEO   = "evap_31"   # video the example drops in Figure 1 come from
EXAMPLE_DROPS   = None        # None = the N_EXAMPLES longest drops; or a list of drop_id strings
N_EXAMPLES      = 6
FILMSTRIP_DROP  = None        # None = first example drop
FILMSTRIP_EVERY = 3           # show every Nth frame of that drop
FILMSTRIP_HALF  = 20          # half-width (px) of each square filmstrip crop
QC_K_MAD        = 3.0         # Figure 3: flag drops whose K is > this many MADs from their video's median
QC_R2_MIN       = 0.90        # Figure 3: flag drops whose d2 fit R^2 (after outlier rejection) is below this
QC_MAX_REJECT   = 0.20        # Figure 3: flag drops where more than this fraction of points were rejected

In [14]:
# ============================ IMPORTS & STYLE ============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

os.makedirs(OUT_DIR, exist_ok=True)

# Fixed categorical order (never cycled). Colour follows the entity: a drop keeps its colour
# wherever it appears in a figure.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, GRID = "#1f1f1e", "#5f5e58", "#e4e3dd"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.titlesize": 11, "axes.titleweight": "regular", "legend.frameon": False,
    "lines.linewidth": 1.6, "lines.markersize": 4.5,
})
print("matplotlib", plt.matplotlib.__version__, "| pandas", pd.__version__)

matplotlib 3.5.1 | pandas 1.4.2


In [15]:
# ======================= LOAD + FILTER =======================
if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f"measurements.csv not found: {CSV_PATH}  (run droplet_size_measurement_v2 first)")
raw = pd.read_csv(CSV_PATH)
if VIDEOS:
    raw = raw[raw["video"].isin(VIDEOS)]
print(f"{len(raw)} rows, {raw['drop_id'].nunique()} drop_ids, videos: {sorted(raw['video'].unique())}\n")

df = raw[raw[SIZE_COL].notna()].copy()
log = [("rows with a measurement", len(df))]

if DROP_FIT_FAILED and "fit_failed" in df:
    df = df[~df["fit_failed"].astype(bool)];        log.append(("  after removing fit_failed", len(df)))
n_soft = int((df["sharpness"] < SHARPNESS_MIN).sum())
df = df[df["sharpness"] >= SHARPNESS_MIN];          log.append((f"  after sharpness >= {SHARPNESS_MIN}  ({n_soft} out-of-focus rows)", len(df)))

# frames per drop are counted AFTER the row filters, so a drop is judged on its usable frames
df["n_use"] = df.groupby("drop_id")["frame"].transform("size")
df = df[df["n_use"] >= MIN_FRAMES].copy();          log.append((f"  after drops with >= {MIN_FRAMES} frames", len(df)))
df = df.sort_values(["video", "drop_id", "frame"]).reset_index(drop=True)

# time measured from the first KEPT frame of each drop
df["t_s"] = df["time_s"] - df.groupby("drop_id")["time_s"].transform("min")
df["D2"]  = df[SIZE_COL] ** 2

for what, n in log:
    print(f"{what:<60} {n:6d}")
print()
print(df.groupby("video")["drop_id"].nunique().rename("drops kept").to_frame().T.to_string())
print(f"\n{SIZE_COL}: {df[SIZE_COL].min():.1f} - {df[SIZE_COL].max():.1f} um across kept drops")

FileNotFoundError: measurements.csv not found: r'/Users/jacquelinebaeza-rubio/Downloads/SIMPLE/QuIPS/Paul trap scripts/Drop size measurements/2026.10.12'/results/measurements.csv  (run droplet_size_measurement_v2 first)

In [ ]:
# ======================= d2-LAW FIT PER DROP =======================
# D^2 = D0^2 - K t   ->   straight line in (t, D^2); K = -slope.  The slope does not depend on
# where t = 0 is put, so the arbitrary first-detection time does not bias K.
def _line(t, y):
    (m, b), cov = np.polyfit(t, y, 1, cov=True)
    return m, b, cov

def fit_drop(g):
    """Returns (fit dict, boolean array of which rows of g were used)."""
    use = np.ones(len(g), bool)
    use[:FIT_SKIP_FIRST] = False
    use &= (g[SIZE_COL] >= FIT_MIN_D_UM).to_numpy()
    t, y = g["t_s"].to_numpy(), g["D2"].to_numpy()
    if use.sum() < 4:
        return None, use
    m, b, cov = _line(t[use], y[use])
    if FIT_OUTLIER_K:
        res = y - (m * t + b)
        rs = 1.4826 * np.median(np.abs(res[use] - np.median(res[use])))
        rs = max(rs, 1e-9)
        keep = use & (np.abs(res) <= FIT_OUTLIER_K * rs)
        if keep.sum() >= 4 and keep.sum() < use.sum():
            use = keep
            m, b, cov = _line(t[use], y[use])
    yhat = m * t[use] + b
    ss_res, ss_tot = ((y[use] - yhat) ** 2).sum(), ((y[use] - y[use].mean()) ** 2).sum()
    n_cand = int(len(g) - FIT_SKIP_FIRST)
    return ({"K_um2_per_s": -m, "K_err": float(np.sqrt(cov[0, 0])),
             "D0_fit_um": float(np.sqrt(b)) if b > 0 else np.nan,
             "t_extinct_s": b / -m if m < 0 else np.nan,   # when the fitted D^2 reaches 0
             "r2": 1 - ss_res / ss_tot if ss_tot > 0 else np.nan,
             "rms_D2_um2": float(np.sqrt(ss_res / use.sum())),
             "n_fit": int(use.sum()), "n_rejected": n_cand - int(use.sum())}, use)

rows = []
df["in_fit"] = False
for did, g in df.groupby("drop_id", sort=False):
    f, use = fit_drop(g)
    df.loc[g.index, "in_fit"] = use
    if f is None:
        continue
    rows.append({"drop_id": did, "video": g["video"].iat[0],
                 "first_frame": int(g["frame"].iat[0]), "last_frame": int(g["frame"].iat[-1]),
                 "n_frames": len(g), "duration_s": g["t_s"].iat[-1],
                 "D_first_um": g[SIZE_COL].iat[0], "D_last_um": g[SIZE_COL].iat[-1],
                 "median_aspect": g["aspect"].median() if "aspect" in g else np.nan, **f})
fits = pd.DataFrame(rows)
fits.to_csv(os.path.join(OUT_DIR, "drop_fits.csv"), index=False)

summ = fits.groupby("video").agg(drops=("drop_id", "size"),
                                 K_median=("K_um2_per_s", "median"),
                                 K_q25=("K_um2_per_s", lambda s: s.quantile(.25)),
                                 K_q75=("K_um2_per_s", lambda s: s.quantile(.75)),
                                 D_first_median=("D_first_um", "median"),
                                 r2_median=("r2", "median"))
print("wrote", os.path.join(OUT_DIR, "drop_fits.csv"), f"({len(fits)} drops)\n")
print(summ.round(2).to_string())
print(f"\n{int(fits['n_rejected'].sum())} of {int(fits['n_fit'].sum() + fits['n_rejected'].sum())} points rejected as single-frame outliers")
print(f"\nall videos: K = {fits['K_um2_per_s'].median():.0f} um^2/s "
      f"(IQR {fits['K_um2_per_s'].quantile(.25):.0f}-{fits['K_um2_per_s'].quantile(.75):.0f})")

In [ ]:
# ======================= FIGURE 1: example drops =======================
ex_pool = fits[fits["video"] == EXAMPLE_VIDEO]
if EXAMPLE_DROPS:
    ex_ids = list(EXAMPLE_DROPS)
else:
    ex_ids = ex_pool.sort_values(["n_frames", "first_frame"], ascending=[False, True])["drop_id"].head(N_EXAMPLES)
    ex_ids = sorted(ex_ids, key=lambda d: int(fits.set_index("drop_id").at[d, "first_frame"]))
colour = {d: SERIES[i % len(SERIES)] for i, d in enumerate(ex_ids)}   # <= 8 examples: no cycling
fit_of = fits.set_index("drop_id")

def filmstrip(drop_id):
    """Square crops of one drop straight from the video (raw pixels, nearest-neighbour)."""
    g = df[df["drop_id"] == drop_id].iloc[::FILMSTRIP_EVERY]
    path = os.path.join(VIDEO_DIR, g["video"].iat[0] + ".avi")
    if not os.path.isfile(path):
        print(f"(filmstrip skipped: video not found at {path})")
        return None, g
    cap, crops, h = cv2.VideoCapture(path), [], FILMSTRIP_HALF
    for _, r in g.iterrows():
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(r["frame"]))
        ok, im = cap.read()
        if not ok:
            continue
        im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        im = cv2.copyMakeBorder(im, h, h, h, h, cv2.BORDER_REPLICATE)
        cx, cy = int(round(r["centroid_x"])) + h, int(round(r["centroid_y"])) + h
        crops.append(im[cy - h:cy + h + 1, cx - h:cx + h + 1])
    cap.release()
    return (np.hstack([np.pad(c, ((0, 0), (0, 3)), constant_values=255) for c in crops]) if crops else None), g

strip_id = FILMSTRIP_DROP or (ex_ids[0] if len(ex_ids) else None)
strip, strip_rows = filmstrip(strip_id) if strip_id else (None, None)

fig = plt.figure(figsize=(12, 6.6 if strip is not None else 4.6), layout="constrained")
gs = fig.add_gridspec(2 if strip is not None else 1, 2, height_ratios=[1, 3.2] if strip is not None else [1])
if strip is not None:
    axs = fig.add_subplot(gs[0, :])
    lo, hi = np.percentile(strip, [1, 99.5])
    axs.imshow(strip, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest", aspect="equal")
    axs.set_axis_off()
    dt = strip_rows["t_s"].diff().median()
    axs.set_title(f"{strip_id}: every {FILMSTRIP_EVERY}{'rd' if FILMSTRIP_EVERY == 3 else 'th'} frame "
                  f"(~{dt:.2f} s apart), {strip_rows[SIZE_COL].iat[0]:.0f} -> {strip_rows[SIZE_COL].iat[-1]:.0f} um",
                  loc="left", color=INK2, fontsize=10)
axD = fig.add_subplot(gs[-1, 0]); axQ = fig.add_subplot(gs[-1, 1])

for d in ex_ids:
    g, c = df[df["drop_id"] == d], colour[d]
    lab = d.replace(EXAMPLE_VIDEO + "_", "")
    axD.plot(g["t_s"], g[SIZE_COL], "-o", color=c, label=lab, mec="white", mew=0.8)
    gi, go = g[g["in_fit"]], g[~g["in_fit"]]
    axQ.plot(gi["t_s"], gi["D2"], "o", color=c, mec="white", mew=0.8)
    axQ.plot(go["t_s"], go["D2"], "o", mfc="none", mec=c, mew=1.2, ms=6)   # rejected from the fit
    f = fit_of.loc[d]
    tt = np.linspace(g["t_s"].min(), g["t_s"].max(), 50)
    axQ.plot(tt, f["D0_fit_um"] ** 2 - f["K_um2_per_s"] * tt, "-", color=c, lw=1.4, alpha=0.9)

axD.set(xlabel="time since drop appeared (s)", ylabel=f"diameter, {SIZE_COL.replace('_um', '')} (µm)")
axD.set_title(f"{EXAMPLE_VIDEO}: drop diameter vs time", loc="left")
axD.legend(ncol=2, fontsize=9, title="drop", title_fontsize=9)
axQ.set(xlabel="time since drop appeared (s)", ylabel="diameter² (µm²)")
axQ.plot([], [], "o", mfc="none", mec=INK2, mew=1.2, ms=6, label="excluded from fit (outlier)")
axQ.legend(fontsize=9, loc="upper right")
k_ex = fit_of.loc[ex_ids, "K_um2_per_s"]
axQ.set_title(f"d²-law check: straight lines, K = {k_ex.median():.0f} µm²/s (median of these)", loc="left")
for ax in (axD, axQ):
    ax.set_xlim(left=-0.03); ax.set_ylim(bottom=0)
fig.savefig(os.path.join(OUT_DIR, f"fig1_examples_{EXAMPLE_VIDEO}.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ======================= FIGURE 2: all drops =======================
# Left: every drop's D^2, with time shifted so each drop's fitted disappearance is t = 0. If all
# drops evaporate at the same K they collapse onto one line, whatever size they were first seen at.
# Right: the per-drop K, one dot per drop, per video, with the median marked.
vids = sorted(fits["video"].unique())
vcol = {v: SERIES[i % len(SERIES)] for i, v in enumerate(vids)}

fig, (axA, axK) = plt.subplots(1, 2, figsize=(12, 4.6), layout="constrained",
                               gridspec_kw={"width_ratios": [1.6, 1]})
for d, g in df.groupby("drop_id"):
    if d not in fit_of.index or not np.isfinite(fit_of.at[d, "t_extinct_s"]):
        continue
    v = g["video"].iat[0]
    axA.plot(g["t_s"] - fit_of.at[d, "t_extinct_s"], g["D2"], "-", color=vcol[v], lw=0.9, alpha=0.45)
kall = fits["K_um2_per_s"].median()
tline = np.linspace(-fits["t_extinct_s"].max(), 0, 50)
axA.plot(tline, -kall * tline, "--", color=INK, lw=1.3, label=f"median K = {kall:.0f} µm²/s")
for v in vids:
    axA.plot([], [], "-", color=vcol[v], lw=2, label=f"{v} ({(fits['video'] == v).sum()} drops)")
axA.set(xlabel="time relative to fitted disappearance (s)", ylabel="diameter² (µm²)")
axA.set_title("All tracked drops, aligned at disappearance", loc="left")
axA.set_ylim(bottom=0); axA.legend(fontsize=9)

rng = np.random.default_rng(0)
for i, v in enumerate(vids):
    k = fits.loc[fits["video"] == v, "K_um2_per_s"].to_numpy()
    axK.scatter(i + rng.uniform(-0.18, 0.18, len(k)), k, s=22, color=vcol[v], ec="white", lw=0.6, zorder=3)
    med = np.median(k)
    axK.plot([i - 0.3, i + 0.3], [med, med], color=INK, lw=2, zorder=4)
    axK.annotate(f"{med:.0f}", (i + 0.32, med), va="center", fontsize=9, color=INK)
axK.set_xticks(range(len(vids)), vids); axK.grid(axis="x", visible=False)
axK.set(ylabel="K (µm²/s)"); axK.set_title("Evaporation constant per drop", loc="left")
axK.set_xlim(-0.6, len(vids) - 0.2)
fig.savefig(os.path.join(OUT_DIR, "fig2_all_drops.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ======================= FIGURE 3: QC - drops worth a second look =======================
# A wrong drop_id link (two drops stitched together, or one drop split) shows up as a poor
# straight-line fit or a K far from its siblings. These are the drops to open in
# results/qc_tracks/<video>/<drop_id>.png first.
med_k = fits.groupby("video")["K_um2_per_s"].transform("median")
mad_k = fits.groupby("video")["K_um2_per_s"].transform(lambda s: 1.4826 * (s - s.median()).abs().median())
fits["K_outlier"] = (fits["K_um2_per_s"] - med_k).abs() > QC_K_MAD * mad_k
fits["poor_fit"]  = fits["r2"] < QC_R2_MIN
fits["many_rejected"] = fits["n_rejected"] > QC_MAX_REJECT * (fits["n_fit"] + fits["n_rejected"])
flag = fits[fits["K_outlier"] | fits["poor_fit"] | fits["many_rejected"]]

fig, ax = plt.subplots(figsize=(7.5, 4.2), layout="constrained")
for v in vids:
    s = fits[fits["video"] == v]
    ax.scatter(s["D_first_um"], s["K_um2_per_s"], s=24, color=vcol[v], ec="white", lw=0.6, label=v, zorder=3)
ax.scatter(flag["D_first_um"], flag["K_um2_per_s"], s=90, facecolor="none", ec=INK, lw=1.2,
           label="flagged (see table)", zorder=4)
ax.set(xlabel="diameter at first kept frame (µm)", ylabel="K (µm²/s)")
ax.set_title("K should not depend on how big the drop was when first seen", loc="left")
ax.legend(fontsize=9)
fig.savefig(os.path.join(OUT_DIR, "fig3_qc.png"), bbox_inches="tight")
plt.show()

print(f"{len(flag)} of {len(fits)} drops flagged (|K - median| > {QC_K_MAD} MAD, R^2 < {QC_R2_MIN}, "
      f"or > {QC_MAX_REJECT:.0%} of points rejected):")
if len(flag):
    print(flag[["drop_id", "n_frames", "n_rejected", "D_first_um", "D_last_um", "K_um2_per_s", "r2",
                "K_outlier", "poor_fit", "many_rejected"]]
          .round(2).to_string(index=False))
fits.to_csv(os.path.join(OUT_DIR, "drop_fits.csv"), index=False)

## Notes
- Points removed by the outlier step are single frames whose size jumps off the drop's own
  trend (a bad edge fit on that frame). They stay in the CSV, are drawn hollow in Figure 1, and
  `n_rejected` counts them per drop.
- **K** is reported per drop; `drop_fits.csv` also has `K_err`, the standard error from the fit.
  The spread *between* drops is the more honest uncertainty.
- The d²-law (`D²` linear in `t`) is the classic result for diffusion-limited evaporation of a
  drop in still gas. Straight lines in the right-hand panel of Figure 1 are the check that it
  holds here.
- `SIZE_COL = "D_eq_um"` is the circle fit on glare-free edge points. While the laser is on,
  `H_um` reads a little small (see `CONTEXT.md` §3). Re-run with `"V_um"` to see how much K
  depends on that choice.
- `t = 0` is the first frame the drop was *kept*, not when it was made. Drops arrive from the
  dispenser and are first seen at somewhat different sizes. Figure 2 aligns them at the fitted
  disappearance time instead.